# 04) Fine-tune `Salesforce/codet5-base` (LoRA) for patch generation

Input expected from notebook 03: `/kaggle/working/codet5_patch_pairs.csv`

In [ ]:
!pip -q install transformers datasets peft accelerate sentencepiece sacrebleu

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, TaskType, get_peft_model

MODEL_NAME = 'Salesforce/codet5-base'
CSV_PATH = Path('/kaggle/working/codet5_patch_pairs.csv')
OUT_DIR = Path('/kaggle/working/codet5_lora')
MAX_INPUT = 384
MAX_TARGET = 384

assert CSV_PATH.exists(), f'Missing dataset: {CSV_PATH}'

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df.dropna(subset=['vulnerable_code', 'fixed_code'])
df['input_text'] = 'fix vulnerability: ' + df['vulnerable_code']
df['target_text'] = df['fixed_code']

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_ds = Dataset.from_pandas(train_df[['input_text', 'target_text']], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[['input_text', 'target_text']], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[['input_text', 'target_text']], preserve_index=False)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    model_inputs = tokenizer(batch['input_text'], truncation=True, max_length=MAX_INPUT)
    labels = tokenizer(text_target=batch['target_text'], truncation=True, max_length=MAX_TARGET)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_ds = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)
test_ds = test_ds.map(preprocess, batched=True, remove_columns=test_ds.column_names)

In [ ]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q', 'v']
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

In [ ]:
args = Seq2SeqTrainingArguments(
    output_dir=str(OUT_DIR),
    num_train_epochs=2,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    predict_with_generate=True,
    generation_max_length=MAX_TARGET,
    logging_steps=50,
    report_to='none',
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
)

trainer.train()
eval_metrics = trainer.evaluate(test_ds)
eval_metrics

In [ ]:
model.save_pretrained(OUT_DIR / 'adapter')
tokenizer.save_pretrained(OUT_DIR / 'tokenizer')
print('Saved to:', OUT_DIR)